# California Housing Value Prediction — Assignment 3
**Live application:** PASTE YOUR STREAMLIT URL HERE AFTER DEPLOYMENT

Dataset: California Housing Prices (1990), originally derived from US Census records. Kaggle: https://www.kaggle.com/datasets/camnugent/california-housing-prices

Goal: Compare five regression algorithms on all 20,640 observations with 10-fold cross-validation, using RMSE. The values describe district medians in 1990, not individual or current house prices.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.base import clone
import joblib

## 1. Load the full dataset
Download `housing.csv` from Kaggle and upload it when prompted. The notebook intentionally does not generate or substitute data.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert "housing.csv" in uploaded, "Please upload housing.csv from Kaggle"
df = pd.read_csv("housing.csv")
print("Shape:", df.shape)
display(df.head())
display(df.isna().sum().to_frame("Missing"))
assert len(df) == 20640, "Expected 20,640 rows from the original Kaggle dataset"

## 2. Explore the data

In [ ]:
display(df.describe(include="all").T)
fig, ax = plt.subplots(figsize=(7,4))
df["median_house_value"].hist(bins=40, ax=ax)
ax.set(title="Distribution of district median house values", xlabel="Median house value (USD)", ylabel="Districts")
plt.show()

## 3. Prepare the features
Missing bedroom counts are filled using training-fold medians. Numeric features are standardized and ocean proximity is one-hot encoded. Preprocessing is fitted separately within each training fold to avoid leakage.

In [ ]:
target="median_house_value"
X=df.drop(columns=[target]); y=df[target]
numeric=X.select_dtypes(include="number").columns.tolist()
categorical=X.select_dtypes(exclude="number").columns.tolist()
preprocessor=ColumnTransformer([("numeric",Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),numeric),("category",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),categorical)])
print("Numeric:",numeric,"Categorical:",categorical)

## 4. Ten-fold cross-validation
All five algorithms differ from those in the supplied Car Price Prediction example. Lower RMSE indicates better average predictive performance.

In [ ]:
models={"Ridge":Ridge(alpha=10),"Lasso":Lasso(alpha=100,max_iter=4000),"Elastic Net":ElasticNet(alpha=0.1,l1_ratio=0.5,max_iter=4000),"Bayesian Ridge":BayesianRidge(),"Extra Trees":ExtraTreesRegressor(n_estimators=60,min_samples_leaf=3,random_state=42,n_jobs=-1)}
cv=KFold(n_splits=10,shuffle=True,random_state=42)
rows=[]; folds=[]
for name,model in models.items():
    pipeline=Pipeline([("preprocess",clone(preprocessor)),("regressor",clone(model))])
    scores=cross_validate(pipeline,X,y,cv=cv,scoring="neg_root_mean_squared_error",n_jobs=1,return_train_score=False)
    errors=-scores["test_score"]
    rows.append({"Algorithm":name,"Mean RMSE (USD)":errors.mean(),"Std RMSE":errors.std(),"Mean fit time (s)":scores["fit_time"].mean()})
    folds.extend({"Algorithm":name,"Fold":i,"RMSE (USD)":float(value)} for i,value in enumerate(errors,1))
    print(f"{name}: {errors.mean():,.2f} USD")
results=pd.DataFrame(rows).sort_values("Mean RMSE (USD)").reset_index(drop=True)
display(results.round(2))
results.to_csv("comparison_results.csv",index=False)
pd.DataFrame(folds).to_csv("fold_results.csv",index=False)

## 5. Compare results

In [ ]:
ax=results.plot.bar(x="Algorithm",y="Mean RMSE (USD)",legend=False,figsize=(8,4))
ax.set(ylabel="Mean RMSE (USD)",title="10-fold cross-validation comparison")
plt.xticks(rotation=30,ha="right")
plt.tight_layout();plt.show()
print("Lowest mean RMSE:",results.loc[0,"Algorithm"])

## 6. Train and save the selected model
After the cross-validation comparison, fit the best-performing pipeline on the full dataset for deployment. Cross-validation RMSE is an estimate of generalization error, not a guarantee for new regions or modern house prices.

In [ ]:
best_name=results.loc[0,"Algorithm"]
final_model=Pipeline([("preprocess",clone(preprocessor)),("regressor",clone(models[best_name]))])
final_model.fit(X,y)
joblib.dump(final_model,"housing_model.joblib")
print("Saved housing_model.joblib using",best_name)
files.download("housing_model.joblib")
files.download("comparison_results.csv")
files.download("fold_results.csv")

## 7. Notes
The dataset is cross-sectional census-based housing data, not a chronological time series. Random K-fold CV is used to match the assignment instructions; geographically separated validation would provide a stricter test for deployment in unseen areas. Some house values are capped in the source data, which limits prediction accuracy. The deployed tool is for educational use.